# Train Shared Vision Backbone (Ball + Marker CNN)
This notebook clones the repository, pulls the merged `shared_vision` gold dataset via DVC from the home server, and trains the ~70K-param Shared Encoder Backbone (ball regression head + marker segmentation head + marker heatmap head) defined in `train_cnn_2d_tracker_marker.py`.

In [ ]:
EVAL_DATASET_NAME = 'shared_vision'              # Dataset 8, 100% real -- used ONLY for evaluation below
TRAIN_DATASET_NAME = 'shared_vision_synthetic_mix'  # Dataset 9, 60% synthetic / 40% real -- used ONLY for training
VERSION = 'v2'  # v1 was trained on mislabeled ball positions (point-reflected on both axes --
                 # see docs/PROJECT_LOGBOOK.md, 2026-08-12). v2 has never actually been trained/
                 # uploaded yet, so it stays v2 here even though the data backing it now also
                 # includes Dataset 9 (60/40 synthetic/real marker mix, extends
                 # implementation_plan_shared_backbone_cnn.md's Component 3 with shape/color
                 # variation, not just position) -- see docs/PROJECT_LOGBOOK.md, 2026-08-12
                 # (Dataset 9 entry).

**Datasets are now fetched via DVC from the home server (MinIO over Tailscale), not manual Drive upload** — see `docs/DATA_STORAGE.md` and the home_server repo's `docs/COLAB_SETUP.md` for how this works. Nothing to zip/upload manually before running; the bootstrap cells below join the tailnet and `dvc pull` exactly the two dataset subfolders this notebook needs (`EVAL_DATASET_NAME`, `TRAIN_DATASET_NAME`).

One-time setup this requires in Colab's **Secrets** panel (key icon, left sidebar):
- `TAILSCALE_AUTHKEY` — a reusable, ephemeral auth key from `https://login.tailscale.com/admin/settings/keys`
- `MINIO_ACCESS_KEY_ID` / `MINIO_SECRET_ACCESS_KEY` — get these from whoever administers the home server (they are not stored in any repo)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf /content/ball_balance_video_controlled
!git clone https://github.com/Jack0468/ball_balance_video_controlled.git
!pip install pandas torch torchvision albumentations opencv-python-headless matplotlib onnx onnxscript
import torch
print(f"Setup complete. Using torch {torch.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'})")

In [ ]:
# Bootstrap Tailscale (userspace-networking mode, since Colab's sandbox has no TUN
# device permission) so this session can reach the home server's MinIO the same way a
# laptop does. See home_server/docs/COLAB_SETUP.md for the full explanation.
!pip install -q "dvc[s3]" pysocks
!curl -fsSL https://tailscale.com/install.sh | sh

import subprocess
subprocess.Popen(
    ["tailscaled", "--tun=userspace-networking", "--socks5-server=localhost:1055"],
    stdout=open("/content/tailscaled.log", "w"), stderr=subprocess.STDOUT,
)

In [ ]:
from google.colab import userdata
import os, subprocess

authkey = userdata.get("TAILSCALE_AUTHKEY")
subprocess.run(["tailscale", "up", f"--authkey={authkey}", "--hostname=colab-shared-vision-backbone"], check=True)
subprocess.run(["tailscale", "status"])

# socks5h (not socks5) -- the trailing h also routes DNS through the proxy, needed if
# the MagicDNS name is ever used instead of the raw Tailscale IP.
os.environ["ALL_PROXY"] = "socks5h://localhost:1055"
os.environ["HTTPS_PROXY"] = "socks5h://localhost:1055"
os.environ["HTTP_PROXY"] = "socks5h://localhost:1055"

# Credentials from Colab Secrets, not hardcoded -- Colab runtimes are ephemeral so
# there's no persistent .dvc/config.local to protect, but they still shouldn't sit in
# the notebook's saved source/output.
%cd /content/ball_balance_video_controlled
!dvc remote modify --local homeserver access_key_id {userdata.get("MINIO_ACCESS_KEY_ID")}
!dvc remote modify --local homeserver secret_access_key {userdata.get("MINIO_SECRET_ACCESS_KEY")}

# Pull exactly the two dataset subfolders this notebook needs, not the whole 03_gold tree.
!dvc pull host_software/data/03_gold/{EVAL_DATASET_NAME} host_software/data/03_gold/{TRAIN_DATASET_NAME}
print("Datasets pulled via DVC!")

### Training
`--resume` is always passed below: `output-dir` is on Google Drive, so if the Colab runtime disconnects or crashes mid-training, `shared_vision_backbone_resume.pt` (model + optimizer + scheduler state + loss history, saved after every epoch) survives there. Just re-run this cell -- it picks up from the last completed epoch instead of retraining from scratch. If no checkpoint exists yet (first run), `--resume` is a no-op and training starts fresh.

In [ ]:
%cd /content/ball_balance_video_controlled
# Run as a module (-m), not a plain script -- the trainer imports
# host_software.ml_vision.training.{shared_vision_dataset,augmentations} as package-qualified
# paths, which only resolve when the repo root is on sys.path (i.e. invoked via -m from here).
!python -m host_software.ml_vision.training.train_cnn_2d_tracker_marker \
    --csv-file host_software/data/03_gold/{TRAIN_DATASET_NAME}/labels.csv \
    --images-dir host_software/data/03_gold/{TRAIN_DATASET_NAME}/images \
    --mask-dir host_software/data/03_gold/{TRAIN_DATASET_NAME}/masks \
    --output-dir /content/drive/MyDrive/VRI_Models/shared_vision_backbone_{VERSION} \
    --resume

### Training Results
`train_cnn_2d_tracker_marker.py` exports the best checkpoint to ONNX and writes a per-session validation breakdown and loss curve.

In [ ]:
from IPython.display import Image, display
import pandas as pd

output_dir = f'/content/drive/MyDrive/VRI_Models/shared_vision_backbone_{VERSION}'
display(Image(filename=f'{output_dir}/training_curve.png'))
pd.read_csv(f'{output_dir}/per_session_eval.csv')

### Evaluation
Runs `evaluate_shared_vision_backbone.py` against `EVAL_DATASET_NAME` (Dataset 8, all-real) -- deliberately **not** the `TRAIN_DATASET_NAME` mix training just used, so reported metrics are never computed against synthetic frames. `evaluate_shared_vision_backbone.py` independently re-derives its own held-out temporal slice (same `--val-fraction` default) from that real-only CSV, disjoint from whatever training used, whether or not the two datasets overlap. Reports ball-position pixel error, marker mask IoU/Dice, heatmap MSE, and inference latency -- metrics `train_cnn_2d_tracker_marker.py` doesn't compute. Also saves an error-distribution histogram and a qualitative grid of predicted-vs-ground-truth ball points and mask contours.

In [ ]:
%cd /content/ball_balance_video_controlled
# Deliberately EVAL_DATASET_NAME (all-real, Dataset 8), not TRAIN_DATASET_NAME --
# reported metrics must never be computed against synthetic frames.
!python -m host_software.ml_vision.evaluations.evaluate_shared_vision_backbone \
    --csv-file host_software/data/03_gold/{EVAL_DATASET_NAME}/labels.csv \
    --images-dir host_software/data/03_gold/{EVAL_DATASET_NAME}/images \
    --mask-dir host_software/data/03_gold/{EVAL_DATASET_NAME}/masks \
    --checkpoint {output_dir}/shared_vision_backbone_best.pt

In [ ]:
import json

display(Image(filename=f'{output_dir}/evaluation_error_histogram.png'))
display(Image(filename=f'{output_dir}/evaluation_visual_grid.png'))
with open(f'{output_dir}/evaluation_metrics.json') as f:
    print(json.dumps(json.load(f), indent=2))